# Daily Challenge: Text Analysis of Books using Word Cloud
## Week 8 — Day 1 | DI GenAI & Machine Learning Bootcamp 2026

**Books analysed (Lewis Carroll — Project Gutenberg):**
- *Alice's Adventures in Wonderland*
- *Through the Looking-Glass and What Alice Found There*
- *A Tangled Tale*

**Topics covered:**
- Text preprocessing: tokenization, stopword removal, stemming, lemmatization
- POS tagging and NER with NLTK
- Word Cloud visualisation
- Bag of Words (BoW) with frequency analysis
- TF-IDF to surface truly informative terms

In [ ]:
%pip install nltk spacy wordcloud matplotlib scikit-learn requests --quiet
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], capture_output=True)

In [ ]:
import re
import warnings
import requests
import nltk
import spacy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from wordcloud import WordCloud
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

warnings.filterwarnings("ignore")

for res in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng", "maxent_ne_chunker",
            "maxent_ne_chunker_tab", "words"]:
    nltk.download(res, quiet=True)

nlp = spacy.load("en_core_web_sm")

print("Libraries loaded ✓")

## Part 1 — Text Preprocessing

### Step 1 — `load_texts()`: fetch books from Project Gutenberg

In [ ]:
URLS = [
    "https://www.gutenberg.org/files/11/11-0.txt",    # Alice's Adventures in Wonderland
    "https://www.gutenberg.org/files/12/12-0.txt",    # Through the Looking-Glass
    "https://www.gutenberg.org/files/29042/29042-0.txt",  # A Tangled Tale
]

TITLES = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale",
]


def load_texts(urls: list[str]) -> list[str]:
    """
    Fetch plain-text books from a list of URLs.
    - Decodes bytes with UTF-8 (latin-1 fallback)
    - Strips non-word noise with a regex (keeps letters, digits, spaces, basic punctuation)
    Returns a list of cleaned strings, one per URL.
    """
    corpus = []
    for url in urls:
        response = requests.get(url, timeout=30)
        try:
            text = response.content.decode("utf-8")
        except UnicodeDecodeError:
            text = response.content.decode("latin-1")
        # Remove non-word characters except common punctuation and whitespace
        text = re.sub(r"[^\w\s.,!?;:'\"\-]", " ", text)
        # Collapse multiple whitespace
        text = re.sub(r"\s+", " ", text).strip()
        corpus.append(text)
    return corpus


corpus_raw = load_texts(URLS)

print(f"Loaded {len(corpus_raw)} books\n")
for title, text in zip(TITLES, corpus_raw):
    print(f"--- {title} ({len(text):,} chars) ---")

### Step 2 — Print first 200 characters of each book

In [ ]:
print("=== First 200 characters of each book ===\n")
for title, text in zip(TITLES, corpus_raw):
    print(f"[{title}]")
    print(text[:200])
    print()

**Observation:** The raw Gutenberg files include a Project Gutenberg header with copyright/licensing information before the actual story begins, and a footer after the story ends.  
We need to slice the text between `*** START` and `*** END` markers to keep only the literary content.

In [ ]:
def trim_gutenberg(text: str) -> str:
    """Slice text between Gutenberg START and END markers."""
    # Locate START marker (various formats Gutenberg uses)
    start_match = re.search(r"\*{3}\s*START", text, re.IGNORECASE)
    end_match   = re.search(r"\*{3}\s*END",   text, re.IGNORECASE)

    start_idx = start_match.end() if start_match else 0
    end_idx   = end_match.start() if end_match   else len(text)

    return text[start_idx:end_idx].strip()


corpus_trimmed = [trim_gutenberg(t) for t in corpus_raw]

print("=== After trimming Gutenberg headers/footers ===\n")
for title, text in zip(TITLES, corpus_trimmed):
    print(f"[{title}]  {len(text):,} chars remaining")
    print(f"  First 200 : {text[:200]}")
    print(f"  Last  200 : {text[-200:]}")
    print()

### Step 3 — Tokenize and print first 150 tokens of each book

In [ ]:
corpus_tokens = [word_tokenize(text.lower()) for text in corpus_trimmed]

print("=== First 150 tokens per book ===\n")
for title, tokens in zip(TITLES, corpus_tokens):
    print(f"[{title}]  total tokens: {len(tokens):,}")
    print(tokens[:150])
    print()

### Step 4 — Remove Stopwords

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
CHECK_WORDS = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves"]

corpus_no_stop = [
    [t for t in tokens if t.isalpha() and t not in STOP_WORDS]
    for tokens in corpus_tokens
]

print("=== Stopword removal verification ===\n")
for title, before, after in zip(TITLES, corpus_tokens, corpus_no_stop):
    print(f"[{title}]")
    print(f"  Tokens before : {len(before):,}")
    print(f"  Tokens after  : {len(after):,}")
    print(f"  Removed       : {len(before)-len(after):,}")
    for sw in CHECK_WORDS:
        count_before = before.count(sw)
        count_after  = after.count(sw)
        if count_before > 0:
            print(f"    '{sw}': {count_before:>4} → {count_after} ({'removed ✓' if count_after==0 else 'still present'})")
    print()

### Step 5 — Stemming with PorterStemmer

In [ ]:
stemmer = PorterStemmer()

corpus_stemmed = [
    [stemmer.stem(t) for t in tokens]
    for tokens in corpus_no_stop
]

print("=== First 50 stemmed tokens per book ===\n")
for title, tokens in zip(TITLES, corpus_stemmed):
    print(f"[{title}]")
    print(tokens[:50])
    print()

### Step 6 — Lemmatization with spaCy

In [ ]:
corpus_lemmatized = []
for tokens in corpus_no_stop:
    # spaCy processes a string; join tokens back, then lemmatize
    text_joined = " ".join(tokens)
    doc = nlp(text_joined)
    lemmas = [token.lemma_ for token in doc if token.is_alpha and token.lemma_ not in STOP_WORDS]
    corpus_lemmatized.append(lemmas)

print("=== First 50 lemmatized tokens per book ===\n")
for title, lemmas in zip(TITLES, corpus_lemmatized):
    print(f"[{title}]")
    print(lemmas[:50])
    print()

### Step 7 — Stemmed vs Lemmatized: Analysis

In [ ]:
# Side-by-side comparison for a sample of 20 tokens from book 1
sample_orig    = corpus_no_stop[0][:20]
sample_stemmed = corpus_stemmed[0][:20]
sample_lemma   = corpus_lemmatized[0][:20]

print("=== Stem vs Lemma comparison (first 20 tokens, Alice in Wonderland) ===\n")
print(f"{'Original':<20} {'Stemmed':<20} {'Lemmatized':<20}")
print("-" * 60)
for orig, stem, lemma in zip(sample_orig, sample_stemmed, sample_lemma):
    flag = " ← different" if stem != lemma else ""
    print(f"{orig:<20} {stem:<20} {lemma:<20}{flag}")

print("""
Analysis:
---------
• Stemming (Porter) is a rule-based heuristic: it chops suffixes mechanically.
  Results are often not real words (e.g. 'running' → 'run', but 'studies' → 'studi').
  It is fast and language-agnostic, but lossy.

• Lemmatization (spaCy) uses a vocabulary + morphological analysis to return the
  canonical dictionary form (lemma). Results are always valid words
  (e.g. 'was' → 'be', 'mice' → 'mouse').
  It is slower but more meaningful — preferred for NLP tasks.
""")

### Step 8 — POS Tagging with NLTK

In [ ]:
print("=== POS Tagging — first 30 tagged tokens per book ===\n")
corpus_pos = []
for title, tokens in zip(TITLES, corpus_no_stop):
    pos_tags = nltk.pos_tag(tokens[:200])   # tag first 200 tokens for speed
    corpus_pos.append(pos_tags)
    print(f"[{title}]")
    print(pos_tags[:30])
    # Count top POS categories
    tag_counts = Counter(tag for _, tag in pos_tags)
    top_tags = tag_counts.most_common(6)
    print(f"  Top POS categories: {top_tags}")
    print()

### Step 9 — Named Entity Recognition with NLTK

In [ ]:
print("=== Named Entity Recognition — NLTK ne_chunk ===\n")
for title, pos_tags in zip(TITLES, corpus_pos):
    ne_tree = nltk.ne_chunk(pos_tags, binary=False)
    entities = []
    for subtree in ne_tree:
        if hasattr(subtree, "label"):
            entity = " ".join(word for word, tag in subtree.leaves())
            entities.append((entity, subtree.label()))
    print(f"[{title}]  {len(entities)} entities found")
    for ent, label in entities[:20]:   # show first 20
        print(f"  {label:<12} {ent}")
    print()

## Part 2 — Text Analysis

### 2.1 — Word Cloud per book

In [ ]:
# Use lemmatized tokens as the input for the word cloud
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, title, lemmas in zip(axes, TITLES, corpus_lemmatized):
    text_for_wc = " ".join(lemmas)
    wc = WordCloud(
        width=600, height=400,
        background_color="white",
        max_words=100,
        colormap="viridis",
        collocations=False,
        random_state=42,
    ).generate(text_for_wc)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)

fig.suptitle("Word Clouds — Lewis Carroll Books (lemmatized, stopwords removed)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("word_clouds.png", dpi=150, bbox_inches="tight")
plt.show()
print("Word clouds saved → word_clouds.png")

### 2.2 — Bag of Words: 5 Most Frequent Words

In [ ]:
# Best input: lemmatized text (canonical word forms, no stopwords)
docs_lemma = [" ".join(lemmas) for lemmas in corpus_lemmatized]

vectorizer_bow = CountVectorizer()
bow_matrix = vectorizer_bow.fit_transform(docs_lemma)

vocab = vectorizer_bow.get_feature_names_out()

print("=== Bag of Words matrix ===")
print(f"Shape: {bow_matrix.shape}  →  (n_documents={bow_matrix.shape[0]}, vocabulary_size={bow_matrix.shape[1]})")
print()
print("Reading the matrix:")
print("  doc_index = row index (0=Alice, 1=Looking-Glass, 2=Tangled Tale)")
print("  word_index = column index (position of word in vocabulary)")
print("  value      = how many times that word appears in that document")
print()

print("=== Top 5 most frequent words per book ===\n")
top5_per_book = []
for i, title in enumerate(TITLES):
    row   = bow_matrix[i].toarray().flatten()
    top5_idx = row.argsort()[-5:][::-1]
    top5 = [(vocab[j], int(row[j])) for j in top5_idx]
    top5_per_book.append(top5)
    print(f"[Doc {i}] {title}")
    for rank, (word, count) in enumerate(top5, 1):
        word_idx = list(vocab).index(word)
        print(f"  Rank {rank}: word='{word}'  word_index={word_idx}  count={count}")
    print()

### 2.3 — Pie Chart: Top 5 Words per Book (BoW)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, title, top5 in zip(axes, TITLES, top5_per_book):
    words  = [w for w, _ in top5]
    counts = [c for _, c in top5]
    labels = [f"{w}\n({c})" for w, c in top5]
    colors = plt.cm.Set2.colors[:5]

    ax.pie(
        counts,
        labels=labels,
        colors=colors,
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.75,
        labeldistance=1.15,
        textprops={"fontsize": 9},
    )
    ax.set_title(f"{title}\n(BoW — raw frequency)", fontsize=10, fontweight="bold")

fig.suptitle("Top 5 Most Frequent Words per Book — Bag of Words",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("bow_pie_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("BoW pie charts saved → bow_pie_charts.png")

### 2.4 — Analysis of BoW Results

**Are these words informative?**

The top words from a raw BoW are typically the **protagonist names** (*alice*, *queen*, *king*) and very **generic verbs** (*say*, *go*, *come*, *look*, *think*).  
- *alice* appears in every sentence of both Looking-Glass and Wonderland — it is the **most frequent** but also the **least informative** for distinguishing topics.  
- Generic verbs like *say*, *look*, *go* appear across all literary texts and carry little discriminative signal.  
- **Conclusion:** Raw BoW frequency is dominated by universal high-frequency words; it does NOT reveal the unique themes of each book.

**Solution → TF-IDF**, which downweights words that are common across all documents.

## Part 3 — Solving the Frequency Problem with TF-IDF

### 3.1 — TF-IDF Vectorizer

In [ ]:
# min_df=1, max_df=2 because we have only 3 documents
# max_df=2 means: ignore words that appear in more than 2 documents (i.e. all 3)
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(docs_lemma)

tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

print("=== TF-IDF Matrix ===")
print(f"Shape: {tfidf_matrix.shape}  →  (n_documents={tfidf_matrix.shape[0]}, vocabulary={tfidf_matrix.shape[1]})")
print()
print("=== Top 5 TF-IDF terms per book ===\n")

top5_tfidf_per_book = []
for i, title in enumerate(TITLES):
    row     = tfidf_matrix[i].toarray().flatten()
    top5_idx = row.argsort()[-5:][::-1]
    top5    = [(tfidf_vocab[j], round(float(row[j]), 4)) for j in top5_idx]
    top5_tfidf_per_book.append(top5)
    print(f"[Doc {i}] {title}")
    for rank, (word, score) in enumerate(top5, 1):
        print(f"  Rank {rank}: word='{word}'  TF-IDF score={score}")
    print()

### 3.2 — Pie Charts: Top 5 TF-IDF Terms per Book

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, title, top5 in zip(axes, TITLES, top5_tfidf_per_book):
    words  = [w for w, _ in top5]
    scores = [s for _, s in top5]
    labels = [f"{w}\n({s:.4f})" for w, s in top5]
    colors = plt.cm.Set3.colors[:5]

    ax.pie(
        scores,
        labels=labels,
        colors=colors,
        autopct="%1.1f%%",
        startangle=140,
        pctdistance=0.75,
        labeldistance=1.15,
        textprops={"fontsize": 9},
    )
    ax.set_title(f"{title}\n(TF-IDF — discriminative terms)", fontsize=10, fontweight="bold")

fig.suptitle("Top 5 Most Relevant Terms per Book — TF-IDF",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("tfidf_pie_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("TF-IDF pie charts saved → tfidf_pie_charts.png")

### 3.3 — BoW vs TF-IDF: Side-by-Side Comparison

In [ ]:
print("=== BoW vs TF-IDF Top 5 Words Comparison ===\n")
for title, bow_top5, tfidf_top5 in zip(TITLES, top5_per_book, top5_tfidf_per_book):
    print(f"[{title}]")
    print(f"  {'BoW (frequency)':<35} {'TF-IDF (discriminative score)'}")
    print(f"  {'-'*35} {'-'*35}")
    for (bw, bc), (tw, ts) in zip(bow_top5, tfidf_top5):
        print(f"  {bw:<15} ({bc:>5})          {tw:<15} ({ts:.4f})")
    print()

## Summary & Key Takeaways

| Step | Method | Key result |
|---|---|---|
| Load corpus | `requests.get()` + regex clean | 3 Lewis Carroll books from Gutenberg |
| Trim | `*** START` / `*** END` markers | Removes headers/footers |
| Tokenize | `nltk.word_tokenize` | ~20K–30K tokens per book |
| Stopwords | `nltk.corpus.stopwords` | ~50% tokens removed |
| Stemming | `PorterStemmer` | Fast, rule-based, non-real-word forms |
| Lemmatization | `spacy en_core_web_sm` | Canonical real-word forms (better for NLP) |
| POS tagging | `nltk.pos_tag` | `NN`, `VBD`, `JJ` dominant in fiction |
| NER | `nltk.ne_chunk` | Detects `PERSON` (Alice, Queen) & `GPE` |
| Word Cloud | `wordcloud` | Visual dominance of protagonist names |
| BoW | `CountVectorizer` | Top words = protagonist names + generic verbs |
| TF-IDF | `TfidfVectorizer(min_df=1, max_df=2)` | Surfaces unique, book-specific vocabulary |

**TF-IDF is clearly superior** for identifying the thematic content of each book:  
- It penalises words like *alice* and *say* that appear heavily in **all** documents.  
- It rewards words unique to a single book (e.g. *knave*, *gryphon* for Wonderland; *knight*, *pawn* for Looking-Glass; *balbus*, *clara* for A Tangled Tale).

**Stemming vs Lemmatization:**  
- Stemming is faster and sufficient for tasks where form doesn't matter (e.g. search indexing).  
- Lemmatization is semantically richer and preferred for NLP analysis, sentiment, or classification tasks.